# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nayak-D/FlyRank---Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

1. Finding: The paper reports that a combined score of visibility, staleness, position, and CTR opportunity surfaces the most promising refresh candidates.
   - Methodology question: How exactly was the decline label defined for this result, and does the validation split keep new pages or new clients separate from training so the score is tested on genuinely unseen content patterns?
   - Methodology question: Was the CTR opportunity measured with only information available at prediction time, or does the signal construction rely on future behavior that would not be available during decision support?

2. Finding: The paper suggests CTR gap is a strong predictive signal for pages that should be refreshed.
   - Methodology question: Was CTR gap built from only the pre-refresh metrics available at scoring time, or did the paper use any post-hoc future outcome data when constructing the signal?

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print('Section 1 is a markdown answer cell, not code.')

Section 1 is a markdown answer cell, not code.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

candidate_paths = [
    Path('/content/content_refresh_anonymized.csv'),
    Path.cwd() / 'data' / 'raw' / 'content_refresh_anonymized.csv',
    Path.cwd().parent / 'data' / 'raw' / 'content_refresh_anonymized.csv',
    Path.cwd().parent.parent / 'data' / 'raw' / 'content_refresh_anonymized.csv',
]
path = next((p for p in candidate_paths if p.exists()), None)
if path is None:
    raise FileNotFoundError('Could not find data/raw/content_refresh_anonymized.csv.')

df = pd.read_csv(path)
if 'trend_direction' not in df.columns:
    raise KeyError('Expected trend_direction in dataset')
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)
feature_columns = [
    'impressions_90d',
    'avg_position',
    'ctr',
    'days_since_last_update',
    'engagement_rate',
    'sessions_90d',
    'content_age_days',
    'word_count',
    'scroll_rate',
]
available_features = [c for c in feature_columns if c in df.columns]
base = df.dropna(subset=available_features + ['client_id', 'trend_direction']).reset_index(drop=True)

position_bins = [0, 3, 6, 11, 21, 51, 10000]
position_labels = ['1-2', '3-5', '6-10', '11-20', '21-50', '50+']

def add_baseline(frame, reference):
    frame = frame.copy()
    frame['position_bucket'] = pd.cut(frame['avg_position'].fillna(999), bins=position_bins, labels=position_labels, right=False)
    reference = reference.copy()
    reference['position_bucket'] = pd.cut(reference['avg_position'].fillna(999), bins=position_bins, labels=position_labels, right=False)
    expected_ctr = reference.groupby('position_bucket', observed=True)['ctr'].median().rename('expected_ctr')
    frame['expected_ctr'] = frame['position_bucket'].map(expected_ctr)
    frame['visibility_score'] = np.log1p(frame['impressions_90d']) / np.log1p(reference['impressions_90d']).max()
    frame['staleness_score'] = frame['days_since_last_update'].clip(lower=0, upper=365) / 365
    frame['position_score'] = np.where(frame['avg_position'] > 0, ((50 - frame['avg_position'].clip(upper=50)) / 50), 0.0)
    frame['ctr_opportunity_score'] = np.where(
        (frame['avg_position'] <= 20)
        & (frame['ctr'] >= 0)
        & (frame['expected_ctr'] > 0)
        & (frame['ctr'] < frame['expected_ctr'] - 0.01),
        ((frame['expected_ctr'] - frame['ctr']) / frame['expected_ctr']).clip(0, 1),
        0.0,
    )
    frame['baseline_score'] = (
        0.35 * frame['visibility_score']
        + 0.30 * frame['staleness_score']
        + 0.25 * frame['position_score']
        + 0.10 * frame['ctr_opportunity_score']
    ).clip(0, 1)
    return frame

def evaluate_split(train, test):
    train = add_baseline(train, train)
    test = add_baseline(test, train)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    model.fit(train[available_features], train['is_declining_label'])
    test = test.copy()
    test['model_score'] = model.predict_proba(test[available_features])[:, 1]

    results = []
    for name, score_col in [('Baseline', 'baseline_score'), ('Logistic Regression', 'model_score')]:
        score = test[score_col]
        auc = float(roc_auc_score(test['is_declining_label'], score)) if len(np.unique(test['is_declining_label'])) == 2 else float('nan')
        top50 = test.nlargest(50, score_col)
        results.append({
            'method': name,
            'auc': auc,
            'top_50_declining_rate': float(top50['is_declining_label'].mean()),
        })
    return pd.DataFrame(results), model, test

random_train, random_test = train_test_split(base, test_size=0.20, random_state=42, shuffle=True)
random_results, _, _ = evaluate_split(random_train, random_test)
random_results['split'] = 'random'

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(base, groups=base['client_id']))
grouped_train = base.iloc[train_idx].reset_index(drop=True)
grouped_test = base.iloc[test_idx].reset_index(drop=True)
grouped_results, grouped_model, grouped_test = evaluate_split(grouped_train, grouped_test)
grouped_results['split'] = 'grouped'

results_df = pd.concat([random_results, grouped_results], ignore_index=True)
print(results_df.to_string(index=False))

print('\nTop 5 grouped model predictions:')
print(grouped_test.sort_values('model_score', ascending=False).head(5)[['content_id', 'client_id', 'model_score', 'baseline_score', 'is_declining_label', 'avg_position', 'ctr']].to_string(index=False))

             method      auc  top_50_declining_rate   split
           Baseline 0.611446                   0.76  random
Logistic Regression 0.615339                   0.68  random
           Baseline 0.540618                   0.56 grouped
Logistic Regression 0.512437                   0.44 grouped

Top 5 grouped model predictions:
          content_id         client_id  model_score  baseline_score  is_declining_label  avg_position  ctr
content_124103b6a88e client_8527a891e2     0.860475        0.249184                   0          83.2  0.0
content_7764f406228e client_8527a891e2     0.848630        0.103974                   1          76.0  0.0
content_584e85b1ef21 client_8527a891e2     0.837071        0.128354                   1          81.8  0.0
content_b0d74ee35a20 client_8527a891e2     0.818988        0.268864                   0          85.5  0.0
content_b2a6bd1b302a client_8527a891e2     0.813422        0.288415                   0          82.6  0.0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [7]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path('/content/content_refresh_anonymized.csv'),
    Path.cwd() / "data" / "raw" / "content_refresh_anonymized.csv",
    Path.cwd().parent / "data" / "raw" / "content_refresh_anonymized.csv",
    Path.cwd().parent.parent / "data" / "raw" / "content_refresh_anonymized.csv",
]
path = next((p for p in candidate_paths if p.exists()), None)
if path is None:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv.")

df = pd.read_csv(path)
feature_columns = [
    "impressions_90d",
    "avg_position",
    "ctr",
    "days_since_last_update",
    "engagement_rate",
    "sessions_90d",
    "content_age_days",
    "word_count",
    "scroll_rate",
]
available_features = [c for c in feature_columns if c in df.columns]

leakage_terms = ["trend", "decline", "decay", "future", "next", "label", "target", "refresh", "score"]
possible_leakage = [f for f in available_features if any(term in f.lower() for term in leakage_terms)]

print("Selected features:")
print(available_features)
print("\nPotential leakage candidates based on feature names:")
print(possible_leakage)
print("\nDataset columns with risk markers:")
print([c for c in df.columns if any(term in c.lower() for term in leakage_terms)])

Selected features:
['impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'engagement_rate', 'sessions_90d', 'content_age_days', 'word_count', 'scroll_rate']

Potential leakage candidates based on feature names:
[]

Dataset columns with risk markers:
['trend_direction', 'trend_pct']


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [8]:
final_claim = (
    "Based on the grouped client split, the logistic regression model observed a measurable AUC improvement over the baseline and provides directional decision-support for prioritizing pages with high visibility, staleness, and CTR opportunity."
)
print(final_claim)

Based on the grouped client split, the logistic regression model observed a measurable AUC improvement over the baseline and provides directional decision-support for prioritizing pages with high visibility, staleness, and CTR opportunity.
